<a href="https://colab.research.google.com/github/islamsamian22-ship-it/Python-projects/blob/main/Credit%20score%20classifier%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive'))
pd.set_option("display.width", 120)

df = pd.read_csv('/content/drive/MyDrive/train.csv')
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

def clean_numeric(series):

    cleaned = (
        series.astype(str)
        .str.replace("-","", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    cleaned = cleaned.replace({"": None, "nan": None, "!@9#%8": None})
    return pd.to_numeric(cleaned, errors="coerce")

numeric_cols = [ "Age", "Annual_Income", "Num_of_Loan", "Num_of_Delayed_Payment",
    "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt",
    "Amount_invested_monthly", "Monthly_Balance", "Monthly_Inhand_Salary",
    "Total_EMI_per_month", "Credit_Utilization_Ratio", ]

for col in numeric_cols:
    df[col] = clean_numeric(df[col])

df.loc[(df["Age"] < 0) | (df["Age"] > 100), "Age"] = None
df.loc[df["Num_of_Loan"] < 0, "Num_of_Loan"] = None
df.loc[df["Num_of_Delayed_Payment"] < 0, "Num_of_Delayed_Payment"] = None

def parse_credit_history_age(val):
    if pd.isna(val):
        return None
    val = str(val)
    years, months = 0, 0
    if "Year" in val:
        years = int(val.split("Year")[0].strip())
    if "and" in val and "Month" in val:
        months = int(val.split("and")[1].replace("Months", "").strip())
    return years * 12 + months

df["Credit_History_Age_Months"] = df["Credit_History_Age"].apply(parse_credit_history_age)

categorical_cols = ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]
for col in categorical_cols:
    df[col] = df[col].replace({"_______": None, "_": None, "!@9#%8": None})

drop_cols = [
    "ID", "Customer_ID", "Name", "SSN", "Month",
    "Type_of_Loan", "Credit_History_Age",  # replaced by the parsed months version
]
feature_df = df.drop(columns=drop_cols + ["Credit_Score"])
target = df["Credit_Score"]

for col in feature_df.select_dtypes(include=["number"]).columns:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())

for col in feature_df.select_dtypes(include=["object", "string"]).columns:
    feature_df[col] = feature_df[col].fillna(feature_df[col].mode()[0])

label_encoders = {}
for col in feature_df.select_dtypes(include=["object", "string"]).columns:
    le = LabelEncoder()
    feature_df[col] = le.fit_transform(feature_df[col].astype(str))
    label_encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(target)   # Good=0/Poor=1/Standard=2 (alphabetical)

X = feature_df

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n--- Accuracy ---")
print(f"{accuracy_score(y_test, y_pred):.4f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm, index=target_encoder.classes_, columns=target_encoder.classes_))

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n--- Top 10 Most Important Features ---")
print(importances.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Physics Mock QP.docx', '1 Mixture- a substance made when two or more are combined,not chemically bonded.gdoc', '3746_0115876_ORIGSIZE (1).jpg', '3746_0115876_ORIGSIZE.jpg', 'AQA_Biology_Infection and Response KnowIT_GCSE - QUESTIONS-1.docx', 'Itinerary 1028444-1 (1).pdf', 'Itinerary 1028444-1.pdf', 'Abbey DLD International Application Form.pdf', 'Offer Letter ISLAM Samian (1).pdf', 'Contact Form (1).pdf', 'Contact Form.pdf', 'Offer Letter ISLAM Samian.pdf', 'Biomes.pptx', '8464-B-2H-QP-CombinedScienceTrilogy-G-15Jun22-AM.pdf', '8464-C-2H-QP-CombinedScienceTrilogy-G-20Jun22-AM.pdf', '8464-P-2H-QP-CombinedScienceTrilogy-G-23Jun22-AM.pdf', 'Personal statement.gdoc', 'Copy of Personal statement.gdoc', 'Master Copy of Personal statement (1).docx', 'Classroom', 'Samian Islam Resume.docx', 'Copy of Biomes.pptx', 'AQA_Biology_Ecology_KnowIT_GCSE_Questions (1).docx'

/tmp/ipykernel_511/1317477218.py:13: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/train.csv')


Loaded 100000 rows, 28 columns

Train size: 80000, Test size: 20000

--- Accuracy ---
0.7702

--- Classification Report ---
              precision    recall  f1-score   support

        Good       0.69      0.70      0.70      3566
        Poor       0.78      0.76      0.77      5799
    Standard       0.80      0.80      0.80     10635

    accuracy                           0.77     20000
   macro avg       0.75      0.75      0.75     20000
weighted avg       0.77      0.77      0.77     20000


--- Confusion Matrix ---
          Good  Poor  Standard
Good      2510    36      1020
Poor       234  4415      1150
Standard   911  1245      8479

--- Top 10 Most Important Features ---
Outstanding_Debt             0.141563
Interest_Rate                0.099073
Credit_Mix                   0.072167
Delay_from_due_date          0.071507
Changed_Credit_Limit         0.058840
Credit_History_Age_Months    0.052672
Num_Credit_Card              0.048397
Num_Credit_Inquiries         0.042505
T